# ViFinQA — LLM chọn phép tính + dòng (Qwen3-8B)

Chạy trên Colab T4. Đọc `batch_*.jsonl`, ghi `decisions.jsonl`.

**Ràng buộc:** model phải < 14B tham số. Qwen3-8B = 8.2B ✓.
KHÔNG đổi sang Qwen3-14B (~14.7B) — vi phạm thể lệ.

**Bất biến (N7):** model chỉ trả về `operation` + `chosen` (chỉ số vào danh sách ứng viên) + `top_k`. Không bao giờ trả nhãn hay giá trị ô.

In [ ]:
# vllm==0.6.3 predates Qwen3 support entirely (no Qwen3ForCausalLM
# architecture registered) -- LLM(model="Qwen/Qwen3-8B") fails to load.
# Bump to a vLLM release new enough to have Qwen3 support.
!pip -q install "vllm>=0.8.5" "huggingface_hub>=0.24"

In [ ]:
from google.colab import files
import pathlib

pathlib.Path("batches").mkdir(exist_ok=True)
print("Chọn toàn bộ batch_*.jsonl đã sinh bằng `submission row-batches`:")
uploaded = files.upload()
for name in uploaded:
    pathlib.Path("batches", name).write_bytes(uploaded[name])
print("đã nhận:", sorted(p.name for p in pathlib.Path("batches").glob("*.jsonl")))

In [ ]:
from vllm import LLM, SamplingParams

# Design doc (2026-08-22 Sec 5) specifies Qwen3-8B q4 (quantized), not
# full fp16. fp16 Qwen3-8B is ~16.4GB of weights alone -- already over a
# T4's 15GB VRAM budget before any KV cache, so dtype="half" at
# gpu_memory_utilization=0.90 OOMs before the first token.
# Loading a pre-quantized AWQ checkpoint (~5-6GB int4 weights) leaves
# headroom for the KV cache within budget. If this exact repo is
# unavailable, substitute any Qwen3-8B AWQ/GPTQ mirror and keep
# quantization="awq" (or "gptq") to match.
MODEL = "Qwen/Qwen3-8B-AWQ"  # 8.2B params < 14B. KHONG doi sang 14B.
llm = LLM(
    model=MODEL,
    quantization="awq",
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
)
# max_tokens bumped from 16 -> 64 (see cell 4): Qwen3's chat template
# defaults to <think>...</think> reasoning content, which would consume
# the whole budget before any JSON is emitted.
# 64 was sized for a bare {"chosen_index": N} reply. A v2 decision is a
# whole JSON object, and llm.chat() below applies Qwen3's chat template,
# so give the reply room rather than truncating it into unparseable text.
sampling = SamplingParams(temperature=0.0, max_tokens=256)

In [ ]:
# CHAY CELL NAY TRUOC. No in ra text tho cua model cho 3 cau dau tien.
# Muc dich: xac nhan model that su tra ve JSON, truoc khi ton ~1 tieng chay
# ca 16 batch. Lan truoc 1009/1012 quyet dinh la gia tri fallback cua
# parse() chu khong phai model chon -- va khong ai biet cho den khi da chay xong.
import json, pathlib

_probe_batch = sorted(pathlib.Path("batches").glob("batch_*.jsonl"))[0]
_probe = [json.loads(l) for l in _probe_batch.read_text("utf-8").splitlines() if l.strip()][:3]

_outs = llm.chat(
    [[{"role": "user", "content": render(p)}] for p in _probe],
    sampling,
    chat_template_kwargs={"enable_thinking": False},
)

_fallback = 0
for _p, _o in zip(_probe, _outs):
    _text = _o.outputs[0].text
    _decision = parse(_text, len(_p["candidates"]), len(_p.get("companies") or []))
    _is_fallback = _decision == {"operation": "lookup", "chosen": [0], "top_k": None}
    _fallback += _is_fallback
    print("=" * 70)
    print("Q:", _p["question"][:100])
    print("companies:", _p.get("companies"), "| periods:", _p.get("periods"))
    print("--- model tra ve (text tho) ---")
    print(repr(_text[:400]))
    print("--- parse() ra ---", _decision, "<-- FALLBACK" if _is_fallback else "")

print()
if _fallback == len(_probe):
    print("!!! CA", len(_probe), "CAU DEU LA FALLBACK -- DUNG LAI, dung chay cell duoi.")
    print("Xem text tho o tren de biet model dang tra ve cai gi.")
else:
    print("OK:", len(_probe) - _fallback, "/", len(_probe), "cau parse duoc that su. Chay tiep cell duoi.")


In [ ]:
# llm.chat() (not llm.generate()) so vLLM applies Qwen3's chat template
# and the model is actually placed in the assistant role. Measured why:
# with raw completion, 1009/1012 decisions came back as parse()'s exact
# default triple (lookup, [0], None) -- an un-templated instruct model
# echoes the instructions, including the literal JSON skeleton
# {"operation": "...", ...}, which is not valid JSON, so every reply fell
# through to the fallback and the whole run silently produced the rank-1
# baseline. chat_template_kwargs disables Qwen3's default thinking mode,
# which is unreachable from the completion API.
#
# Payload v2 (spec 2026-08-23 Sec 6.2): the batch now also carries
# "companies"/"periods" and a per-candidate "company_code", because the
# model decides the operation, not just a row -- rank/compare_companies
# only make sense when the model can see how many companies are in play.
import json, pathlib, re

_OPERATIONS = {
    "lookup", "compare", "compare_companies", "difference",
    "growth_rate", "ratio", "average", "sum", "rank",
}

PROMPT = """Ban la tro ly phan tich bao cao tai chinh.
Khong giai thich, khong suy luan, khong dung <think>.

Cau hoi: {question}
Cong ty trong cau: {companies}
Ky trong cau: {periods}

Cac dong ung vien:
{candidates}

Chon phep tinh va dong tra loi cau hoi.
- operation la MOT trong: lookup, compare, compare_companies, difference,
  growth_rate, ratio, average, sum, rank
- chosen la danh sach chi so dong. Mot cong ty -> mot chi so.
  Nhieu cong ty (rank, compare_companies, average, sum) -> moi cong ty mot
  chi so, dung thu tu cong ty o tren.
  ratio/compare -> dung hai chi so (tu truoc, mau sau).
- top_k chi can cho rank (1 = cao nhat).

Chi tra ve JSON, khong kem gi khac:
{{"operation": "...", "chosen": [...], "top_k": null}}"""


def render(payload):
    lines = []
    for c in payload["candidates"]:
        parts = [f'[{c["index"]}] {c["row_label"]}']
        if c.get("company_code"):
            parts.append(f'(cong ty: {c["company_code"]})')
        if c.get("row_group_context"):
            parts.append(f'(muc: {c["row_group_context"]})')
        if c.get("table_title"):
            parts.append(f'(bang: {c["table_title"]})')
        if c.get("periods"):
            parts.append(f'(ky: {", ".join(c["periods"])})')
        lines.append(" ".join(parts))
    return PROMPT.format(
        question=payload["question"],
        companies=", ".join(payload.get("companies") or []) or "(khong ro)",
        periods=", ".join(payload.get("periods") or []) or "(khong ro)",
        candidates="\n".join(lines),
    )


def parse(text, limit, n_companies):
    # Neu Qwen3 van ro ri noi dung <think>...</think> (bat chap chi thi
    # trong prompt), bo han doan do truoc khi tim JSON -- neu khong, mot
    # con so bat ky trong phan suy luan co the bi nhat nham lam chi so
    # mot cach im lang, sai ma khong ai biet.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    operation, chosen, top_k = "lookup", [], None
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            payload = json.loads(match.group(0))
            if payload.get("operation") in _OPERATIONS:
                operation = payload["operation"]
            raw = payload.get("chosen")
            if isinstance(raw, list):
                chosen = [int(v) for v in raw if isinstance(v, (int, float))]
            if isinstance(payload.get("top_k"), (int, float)):
                top_k = int(payload["top_k"])
        except (ValueError, TypeError):
            pass
    chosen = [v for v in chosen if 0 <= v < limit] or [0]
    if operation in {"rank", "compare_companies"} and n_companies < 2:
        operation = "lookup"  # arity khong thoa -> de local ha cap
    return {"operation": operation, "chosen": chosen, "top_k": top_k}


out = pathlib.Path("decisions.jsonl")
done = set()
if out.exists():  # chay lai sau timeout chi ton phan con thieu
    done = {json.loads(l)["question_id"] for l in out.read_text("utf-8").splitlines() if l.strip()}
    print("da co san:", len(done))

with out.open("a", encoding="utf-8") as sink:
    for batch in sorted(pathlib.Path("batches").glob("batch_*.jsonl")):
        payloads = [json.loads(l) for l in batch.read_text("utf-8").splitlines() if l.strip()]
        payloads = [p for p in payloads if p["question_id"] not in done and p["candidates"]]
        if not payloads:
            continue
        outputs = llm.chat(
            [[{"role": "user", "content": render(p)}] for p in payloads],
            sampling,
            chat_template_kwargs={"enable_thinking": False},
        )
        for payload, output in zip(payloads, outputs):
            decision = parse(
                output.outputs[0].text,
                len(payload["candidates"]),
                len(payload.get("companies") or []),
            )
            sink.write(json.dumps({"question_id": payload["question_id"], **decision}) + "\n")
        sink.flush()
        print(batch.name, "xong", len(payloads), "cau")


In [ ]:
from google.colab import files
files.download("decisions.jsonl")